
# 🏨 ML Project: Hotel Booking Cancellation Prediction

---

## 1.1) Problem Statement

هتل‌ها به‌طور مداوم با چالش **لغو رزرو (Booking Cancellation)** مواجه هستند. 
لغو رزرو باعث می‌شود:
- ظرفیت هتل به‌درستی مدیریت نشود (Overbooking یا Underbooking)
- درآمد پیش‌بینی‌شده هتل کاهش یابد
- برنامه‌ریزی منابع (نیروی انسانی، غذا، خدمات) دچار خطا شود

اگر بتوان **پیش از موعد** احتمال لغو یک رزرو را پیش‌بینی کرد، هتل می‌تواند:
- سیاست‌های قیمت‌گذاری پویا اعمال کند
- برای رزروهای پرریسک، تاییدیه یا پیش‌پرداخت درخواست کند
- ظرفیت را بهینه مدیریت کند (Overbooking هوشمندانه)

## 1.2) Objective (هدف مدل)

هدف اصلی این پروژه، ساخت یک مدل **Classification** است که با گرفتن اطلاعات یک رزرو 
(در لحظه ثبت رزرو، **نه بعد از آن**)، احتمال لغو شدن آن رزرو را پیش‌بینی کند.

> ⚠️ نکته مهم درباره Data Leakage:
> ستون‌های `reservation_status` و `reservation_status_date` مستقیماً نتیجه نهایی 
> رزرو (Check-Out / Canceled / No-Show) را نشان می‌دهند و در واقع همان Target 
> با شکل دیگری هستند. این دو ستون **باید در مرحله Data Cleaning حذف شوند** 
> چون در لحظه ثبت رزرو (زمانی که می‌خواهیم پیش‌بینی کنیم) این اطلاعات وجود ندارد.

## 1.3) نوع مسئله

**Primary Task → Classification (Binary)**
- Target: `is_canceled`
  - `0` = رزرو لغو نشده (Check-Out)
  - `1` = رزرو لغو شده (Canceled)

**Bonus Task → Regression**
- Target: `adr` (Average Daily Rate - نرخ روزانه اقامت)
- این بخش به‌صورت خلاصه‌تر در انتهای نوت‌بوک ارائه می‌شود.

## 1.4) معیارهای ارزیابی و چرایی انتخاب آن‌ها

با توجه به این‌که (همان‌طور که در EDA نشان داده خواهد شد) توزیع `is_canceled` 
کاملاً متوازن نیست (حدود 63% لغو نشده / 37% لغو شده)، **صرفاً Accuracy کافی نیست.**

معیارهای انتخابی:
| معیار | دلیل انتخاب |
|---|---|
| **F1-Score** | تعادل بین Precision و Recall؛ مناسب برای داده نسبتاً نامتوازن |
| **Recall (Class=1)** | از منظر کسب‌وکار، از دست دادن یک رزرو *واقعاً لغوشونده* (False Negative) پرهزینه‌تر است چون هتل ظرفیت را اشتباه برنامه‌ریزی می‌کند |
| **Precision (Class=1)** | هشدار زیاد و اشتباه (False Positive) هم اعتماد به سیستم را کم می‌کند |
| **ROC-AUC** | معیار کلی برای توانایی مدل در تفکیک دو کلاس، مستقل از threshold |
| **Confusion Matrix** | دیدن دقیق نوع خطاها (FP vs FN) |

معیار اصلی انتخاب بهترین مدل: **F1-Score** (با نگاه ویژه به Recall کلاس ۱)

## 1.5) Dataset Description (خلاصه - جزئیات کامل در بخش بعد)

| ویژگی | مقدار |
|---|---|
| نام | Hotel Booking Demand |
| منبع | Kaggle |
| لینک | https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand |
| تعداد نمونه | 119,390 |
| تعداد ویژگی | 32 |
| Target (Classification) | is_canceled |
| Target (Regression - Bonus) | adr |


## Import Libraries


In [ ]:
#main Libraries

import sys 
import platform


import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    cross_validate, GridSearchCV, RandomizedSearchCV
)

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# --- Feature Selection ---
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif, RFE
)


# --- Models (Classification) ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier



# --- Models (Regression - Bonus Section) ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

# --- Metrics ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)




## settings 

In [ ]:

# --- Settings ---
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.unicode_minus'] = False

# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Environment Info (برای Reproducibility - بخش 28 سند) ---
print("="*60)
print("ENVIRONMENT INFO")
print("="*60)
print(f"Python version     : {platform.python_version()}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")

import sklearn
print(f"Scikit-learn version: {sklearn.__version__}")

import xgboost
print(f"XGBoost version    : {xgboost.__version__}")

import matplotlib
print(f"Matplotlib version : {matplotlib.__version__}")
print(f"Seaborn version    : {sns.__version__}")
print(f"Random State       : {RANDOM_STATE}")
print("="*60)


# [Markdown Cell 2]

### تفسیر
تمام کتابخانه‌های مورد نیاز برای Pipeline کامل پروژه (از EDA تا Tuning) 
بارگذاری شدند. نسخه‌ها برای قابلیت بازتولید پروژه (Reproducibility - بخش ۲۸ سند) 
ثبت شده‌اند. `RANDOM_STATE=42` در تمام مراحل پروژه (Split، مدل‌ها، Cross Validation) 
به‌صورت ثابت استفاده خواهد شد تا نتایج قابل تکرار باشند.

## Load Dataset

In [ ]:
# 3. LOAD DATASET

DATA_PATH = "hotel_bookings.csv"  # مسیر را متناسب با محیط خود تغییر دهید

df_raw = pd.read_csv(DATA_PATH)

# یک نسخه اصلی (Raw) را دست‌نخورده نگه می‌داریم و روی کپی کار می‌کنیم
df = df_raw.copy()

print(f"✅ Dataset با موفقیت بارگذاری شد.")
print(f"تعداد رکوردها : {df.shape[0]:,}")
print(f"تعداد ستون‌ها : {df.shape[1]}")


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("لیست ستون‌ها:\n")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## Data Understanding



## 4. Data Understanding

در این بخش، بدون هیچ‌گونه تغییری در داده، ساختار، انواع داده، آمار توصیفی و 
مقادیر خالی/تکراری را بررسی می‌کنیم تا شناخت کاملی نسبت به Dataset پیدا کنیم.

In [ ]:
# [Code Cell 6]
# --- 4.1) Shape ---
print(f"Shape: {df.shape}")
print(f"تعداد کل داده‌ها (Cells): {df.shape[0] * df.shape[1]:,}")

In [ ]:
# [Code Cell 7]
# --- 4.2) Data Types ---
df.dtypes.to_frame(name='Data Type')

In [ ]:
# [Code Cell 8]
# --- 4.2.1) دسته‌بندی ستون‌ها بر اساس نوع ---
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"📊 ستون‌های عددی ({len(numerical_cols)}):")
print(numerical_cols)
print(f"\n📋 ستون‌های Categorical ({len(categorical_cols)}):")
print(categorical_cols)

In [ ]:
# [Code Cell 9]
# --- 4.3) Statistical Summary - Numerical ---
df.describe().T

In [ ]:
# [Code Cell 10]
# --- 4.3.1) Statistical Summary - Categorical ---
df.describe(include='object').T

In [ ]:
# [Code Cell 11]
# --- 4.4) Missing Values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
}).sort_values(by='Missing Count', ascending=False)

missing_summary[missing_summary['Missing Count'] > 0]

In [ ]:
# [Code Cell 12]
# --- 4.5) Duplicate Rows ---
n_duplicates = df.duplicated().sum()
print(f"تعداد رکوردهای کاملاً تکراری: {n_duplicates:,}")
print(f"درصد از کل داده: {n_duplicates/len(df)*100:.2f}%")

In [ ]:
# [Code Cell 13]
# --- 4.6) Unique Values per Column ---
nunique_df = df.nunique().sort_values(ascending=False).to_frame(name='Unique Values')
nunique_df

In [ ]:
# [Code Cell 14]
# --- 4.7) بررسی دقیق‌تر ستون‌های Categorical (برای شناسایی مقادیر عجیب) ---
for col in categorical_cols:
    print(f"\n🔹 {col} ({df[col].nunique()} مقدار یکتا):")
    print(df[col].value_counts().head(10))
    print("-" * 50)

In [ ]:
# [Code Cell 15]
# --- 4.8) بررسی Target Variable (Classification) ---
print("توزیع is_canceled:")
print(df['is_canceled'].value_counts())
print(f"\nدرصد:")
print(df['is_canceled'].value_counts(normalize=True) * 100)

# [Markdown Cell 4]

## 🔎 تفسیر کامل یافته‌های Data Understanding

### 4.1) حجم داده
Dataset شامل **119,390 رکورد** و **32 ستون** است. این حجم برای یک پروژه ML 
مناسب است (طبق قانون سند: حداقل 1000 نمونه و 8 ویژگی - این شرط با اختلاف زیاد برآورده شده).

### 4.2) انواع داده
- **32 ستون** شامل ترکیبی از:
  - ستون‌های عددی (int64/float64): مثل `lead_time`, `adr`, `stays_in_week_nights`
  - ستون‌های Categorical (object): مثل `hotel`, `meal`, `country`, `market_segment`
- برخی ستون‌های عددی در واقع **مفهوم Categorical** دارند (مثل `is_canceled`, 
  `is_repeated_guest`) که باید در Preprocessing به‌عنوان Binary/Categorical در نظر گرفته شوند.
- ستون `children` از نوع float است در حالی که باید عدد صحیح (تعداد کودک) باشد؛ 
  این به دلیل وجود Missing Value (NaN) در این ستون است که در Pandas باعث تبدیل 
  خودکار به float می‌شود. → باید در Cleaning اصلاح شود.

### 4.3) آمار توصیفی - نکات کلیدی
- `adr` (نرخ روزانه): min = **منفی** (-6.38!) که از نظر منطقی **غیرممکن** است 
  (Invalid Value) و max = **5400** که به‌شدت از میانگین (~101) فاصله دارد ← 
  نشانه قطعی وجود Outlier / داده نامعتبر.
- `lead_time` (فاصله زمانی رزرو تا ورود): از 0 تا 737 روز؛ توزیع احتمالاً به‌شدت 
  Right-Skewed است (اکثر رزروها با فاصله کم، تعداد کمی با فاصله خیلی زیاد).
- `babies` و `children`: حداکثر مقادیری مثل 10 دارند که برای یک رزرو هتل غیرمعمول 
  و مشکوک به خطای داده‌ای است.
- `stays_in_week_nights` و `stays_in_weekend_nights`: برخی رکوردها مقدار **صفر** 
  در هر دو دارند (یعنی صفر شب اقامت) که باید به‌عنوان رکورد نامعتبر بررسی شود.

### 4.4) Missing Values
| ستون | تحلیل |
|---|---|
| `company` (~94% خالی) | اکثر مهمانان به شرکتی وابسته نیستند؛ NaN معنادار = "بدون شرکت" |
| `agent` (~14% خالی) | یعنی رزرو مستقیم بدون آژانس مسافرتی |
| `country` (~0.4% خالی) | گزارش نشدن کشور، به احتمال زیاد Missing at Random |
| `children` (4 رکورد) | به احتمال زیاد سهو در ثبت، قابل جایگزینی با 0 |

### 4.5) Duplicate Rows
تعداد قابل‌توجهی رکورد کاملاً تکراری وجود دارد. با توجه به ماهیت داده (امکان 
وجود رزروهای واقعاً مشابه از مهمانان مختلف)، این تکرارها می‌توانند تا حدی طبیعی 
باشند، اما حجم بالای آن‌ها نشانه احتمالی Duplicate Entry در سیستم ثبت است. 
تصمیم نهایی (حذف یا نگه‌داشتن) در بخش Data Cleaning با توجیه کامل گرفته می‌شود.

### 4.6) Unique Values
- ستون‌هایی مثل `country` (178 کشور) و `agent`/`company` (تعداد زیاد ID) → 
  High Cardinality دارند و برای Encoding نیاز به استراتژی خاص (نه One-Hot ساده) دارند.
- ستون‌های Binary مثل `is_canceled`, `is_repeated_guest` فقط 2 مقدار یکتا دارند.

### 4.7) مقادیر عجیب در Categorical
- ستون `meal` مقدار `"Undefined"` دارد که هم‌معنای `"SC"` (Self Catering / بدون وعده) 
  است ← نمونه بارز **Inconsistent Category** که باید یکسان‌سازی شود.
- ستون `distribution_channel` و `market_segment` مقدار `"Undefined"` دارند 
  (تعداد کم) که باید به‌عنوان Missing در نظر گرفته شوند یا حذف شوند.

### 4.8) Target Variable - is_canceled
- کلاس 0 (لغو نشده): ~63%
- کلاس 1 (لغو شده): ~37%

نسبت تقریبی **63 به 37** نشان می‌دهد داده **کاملاً متوازن نیست** اما به شدت هم 
نامتوازن (Severely Imbalanced) نیست. این سطح از عدم توازن باعث می‌شود:
- **Accuracy به‌تنهایی گمراه‌کننده باشد** (یک مدل که همیشه کلاس 0 را پیش‌بینی 
  کند، Accuracy حدود 63% می‌گیرد بدون یادگیری واقعی)
- نیاز به بررسی دقیق **F1-Score و Recall کلاس ۱** به‌جای تکیه صرف بر Accuracy

این یافته، تصمیم بخش ۱ (استفاده از F1/Recall/ROC-AUC به‌جای صرفاً Accuracy) را 
تایید می‌کند.